In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.constants import hbar, k as kB


# Physical constants
M_K39 = 39 * 1.6605e-27            # Mass of K39 in kg
A0 = 5.29177e-11                  # Bohr radius


# Dimensionless units
length_unit = 1e-6                                           # m
energy_unit = hbar**2 / (M_K39 * length_unit**2)             # J
time_unit   = hbar / energy_unit                             # s
temp_unit   = energy_unit / kB                               # K


# Simulation parameters 
L_phys     = 50e-6                # System size (m)
T_phys     = 310e-9               # Temperature (K)
n0_phys    = 20e12                # Density (m^-2)
omega_z    = 2 * np.pi * 1e3      # Trap frequency (Hz)
dt_phys    = 1e-6                 # Time step (s)
N_grid     = 256                  # Number of point
n_cut      = 2                    # Minimum occupation number (PGPE -> n_cut>>1)
save_every=100

# Initial state
a_ini   = 31 * A0                 # Scattering length (m)
gamma_ini   = 0.03                # Bath coupling  
time_phys_thermalization  = 1 # Time evolutiom (s)                

# Final state
a_final               = 5*a_ini   # Scattering length (m)
gamma_final           = 0         # Bath coupling  (PGPE -> 0)
time_phys_relaxation  =  0.5      # Time evolutiom (s)                


In [2]:
import json
from pathlib import Path
from dataclasses import dataclass

full_path_ini=r"D:\Users\Public\Documents\Louis\Simulations data\Data - relaxation\T=24.92-mu_ini=0.32-n0=20.00-gamma=0.0300-dt=0.00163"


with open(full_path_ini+r"\metadata_initial_state.json", 'r') as file:
    data = json.load(file)


L_phys= data["L_phys"]
T_phys=data["T_phys"]
n0_phys=data["n0_phys"]
omega_z=data["omega_z"]
mu_ini=data["mu_ini"]
L=data["L"]
T=data["T"]
gamma_ini=data["gamma_ini"]
dt=data["dt"]
n_steps=data["n_steps"]
N_grid=data["N_grid"]
k_cut=data["k_cut"]



dx      = L / N_grid
x    = np.linspace(-L/2, L/2, N_grid, endpoint=False)
k    = np.fft.fftfreq(N_grid, d=dx) * 2 * np.pi
KX, KY = np.meshgrid(k, k)
K2   = KX**2 + KY**2
osc_len_z  = np.sqrt(hbar / (M_K39 * omega_z))
n0=n0_phys*length_unit**2
g_ini=mu_ini/n0

projector  = (np.sqrt(K2) <= k_cut).astype(complex)

def init(a_final,n_steps_final,gamma_final,quench_step):

    g_phys_final  = (hbar**2 / M_K39) * np.sqrt(8 * np.pi) * (a_final / osc_len_z)
    g_final    = np.round(g_phys_final / (energy_unit * length_unit**2), 4)
    mu_final = np.round(g_final * n0, 4)
    T_BKT_final= 2 * np.pi * n0 / (np.log(380 / g_final))

    # Simulation stability

    def check_stability(dx, dt, mu_ini, mu_final):
            HEADER   = '\033[95m'
            OKGREEN  = '\033[92m'
            WARNING  = '\033[93m'
            FAIL     = '\033[91m'
            ENDC     = '\033[0m'
            BOLD     = '\033[1m'

            def report_line(label, value, limit, passed):
                status = f"{OKGREEN}[PASS]{ENDC}" if passed else f"{FAIL}[FAIL]{ENDC}"
                print(f"{label:<35} {value:>10.4f} | {limit:>10.4f} | {status}")

            print(f"\n{BOLD}{HEADER}" + "="*75 + f"{ENDC}")
            print(f"{BOLD}SIMULATION DIAGNOSTIC REPORT{ENDC}")
            print(f"{HEADER}" + "="*75 + f"{ENDC}")
            print(f"{BOLD}{'Metric':<35} {'Value':>10} | {'Limit':>10} | {'Status':<10}{ENDC}")
            print("-" * 75)

            kin_val  = dt / (dx**2);  kin_lim = 0.2 / np.pi
            pass_kin = kin_val < kin_lim
            report_line("Kinetic Phase Propagator (dt/dx²)", kin_val, kin_lim, pass_kin)

            nl_val  = max(mu_ini,mu_final) * dt;  nl_lim = 0.1
            pass_nl = nl_val < nl_lim
            report_line("Mean-field Phase Rotation (μ·dt)", nl_val, nl_lim, pass_nl)

            xi       = 1 / np.sqrt(2 * max(mu_ini,mu_final))
            xi_res   = dx / xi;  xi_lim = 0.5
            pass_xi  = xi_res < xi_lim
            report_line("Grid Resolution (dx/ξ)", xi_res, xi_lim, pass_xi)

            print(f"{HEADER}" + "="*75 + f"{ENDC}")
            if not all([pass_kin, pass_nl, pass_xi]):
                print(f"{BOLD}{WARNING}RECOMMENDED SYSTEM TUNING:{ENDC}")
                if not pass_kin or not pass_nl:
                    print("  * Temporal Accuracy: Reduce 'dt'.")
                if not pass_xi:
                    print("  * Spatial Resolution: Increase 'N' or decrease 'L_grid'.")
                print(f"{HEADER}" + "="*75 + f"{ENDC}\n")
            else:
                print(f"{OKGREEN}{BOLD}System parameters are optimal.{ENDC}\n")

    check_stability(dx, dt, mu_ini, mu_final)


    # Parameter recap and PGPE cutoff check
    @dataclass
    class SimParams:
        name: str
        value: float
        unit: str = ""
        fmt: str = ".4f"

    # Collect all data into a structured list
    data = [
        SimParams("L", L),
        SimParams("T", T),
        SimParams("Time (thermal)", time_phys_thermalization/time_unit),
        SimParams("dt", dt, fmt=".6f"),
        SimParams("n_steps", n_steps, fmt=".2e"),
        SimParams("n0", n0, fmt=".6f"),
        SimParams("mu_ini", mu_ini, fmt=".6f"),
        SimParams("mu_final", mu_final, fmt=".6f"),
        SimParams("k_cut", k_cut, fmt=".6f"),
        SimParams("T_BKT_final", T_BKT_final, fmt=".6f"),
    ]

    # Folder creation
    base_dir = Path(full_path_ini+r"\Quench")
    folder_name = f"T={T:.2f}-mu_final={mu_final:.2f}-n0={n0:.2f}-gamma={gamma_final:.4f}-dt={dt}"
    full_path = base_dir / folder_name
    full_path.mkdir(parents=True, exist_ok=True)

    parameters = {
        "L_phys": L_phys,
        "T_phys": T_phys,
        "n0_phys": n0_phys,
        "omega_z": omega_z,
        "mu_ini": mu_ini,
        "mu_final": mu_final,
        "L": L,
        "T": T,
        "gamma_ini": gamma_ini,
        "gamma_final": gamma_final,
        "dt": dt,
        "n_steps_final": n_steps_final,
        "quench_step":quench_step,
        "N_grid": N_grid,
        "k_cut": k_cut,
        "save_every":save_every
    }


    with open(full_path / "metadata_final_state.json", "w") as f:
        json.dump(parameters, f, indent=4)

    print(f"\n--- SAVING ---")
    print(f"Metadata successfully saved to: '{full_path / 'metadata_final_state.json'}'")

    return full_path, g_final



In [3]:
def evolve_SPGPE(psi, g, mu, gamma):
    psi_k = np.fft.fft2(psi) * projector

    V = g * np.abs(psi)**2
    force = (-1j - gamma) * (V - mu) * psi

    noise = (np.random.normal(size=psi.shape)
                + 1j*np.random.normal(size=psi.shape)) / np.sqrt(2)

    noise_scale = np.sqrt(2 * gamma * T * dt / dx**2)
    dW = np.fft.fft2(noise_scale * noise) * projector

    drift = (-1j - gamma) * (0.5 * K2)
    psi_k = (psi_k + np.fft.fft2(force)*projector*dt + dW) / (1 - drift*dt)

    return np.fft.ifft2(psi_k * projector)

## Quench loop

In [15]:
import tqdm

a_finals=np.array([4,1/4])*a_ini
gamma_finals=np.array([gamma_ini,0])
t_final=25 #ms
n_steps_final= int(t_final*1e-3//dt_phys)
quench_step=int(10*1e-3//dt_phys)
run=200


rand_idxs=np.random.randint(1000,n_steps//save_every+1,size=run)
print(n_steps_final,quench_step)

for gamma_final in gamma_finals:
    for a_final in a_finals:
        full_path, g_final=init(a_final,n_steps_final,gamma_final,quench_step)
        
        for r in range(run):
            j=0
            psi_inis = np.memmap(full_path_ini+'\psi_gound_state.dat',
                        dtype=np.complex128,
                        mode='r',
                        shape=(n_steps//save_every+1,N_grid,N_grid))

            psi=psi_inis[rand_idxs[r]]
            del psi_inis


            psi_storage = np.memmap(full_path/f'psi_final_run{r}.dat',
                                    dtype=np.complex128,
                                    mode='w+',
                                    shape=(n_steps_final//save_every+1,N_grid,N_grid))

            for i in tqdm.tqdm(range(n_steps_final)):
                g= g_final if i>quench_step else g_ini
                gamma=gamma_final if i>quench_step else gamma_ini
                mu= g*n0

                psi= evolve_SPGPE(psi, g, mu, gamma)

                if i % save_every ==0:
                    psi_storage[j,:,:] = psi

                    if j%10 ==0:
                        psi_storage.flush()

                    j+=1

            psi_storage.flush()
            del psi_storage


<>:20: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<>:20: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
C:\Users\BEC4-Share\AppData\Local\Temp\ipykernel_22048\788504733.py:20: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
  psi_inis = np.memmap(full_path_ini+'\psi_gound_state.dat',


25000 10000

SIMULATION DIAGNOSTIC REPORT
Metric                                   Value |      Limit | Status    
---------------------------------------------------------------------------
Kinetic Phase Propagator (dt/dx²)       0.0427 |     0.0637 | [PASS]
Mean-field Phase Rotation (μ·dt)        0.0021 |     0.1000 | [PASS]
Grid Resolution (dx/ξ)                  0.3140 |     0.5000 | [PASS]
System parameters are optimal.


--- SAVING ---
Metadata successfully saved to: 'D:\Users\Public\Documents\Louis\Simulations data\Data - relaxation\T=24.92-mu_ini=0.32-n0=20.00-gamma=0.0300-dt=0.00163\Quench\T=24.92-mu_final=1.29-n0=20.00-gamma=0.0300-dt=0.00163\metadata_final_state.json'


100%|██████████| 25000/25000 [07:53<00:00, 52.76it/s]



SIMULATION DIAGNOSTIC REPORT
Metric                                   Value |      Limit | Status    
---------------------------------------------------------------------------
Kinetic Phase Propagator (dt/dx²)       0.0427 |     0.0637 | [PASS]
Mean-field Phase Rotation (μ·dt)        0.0005 |     0.1000 | [PASS]
Grid Resolution (dx/ξ)                  0.1572 |     0.5000 | [PASS]
System parameters are optimal.


--- SAVING ---
Metadata successfully saved to: 'D:\Users\Public\Documents\Louis\Simulations data\Data - relaxation\T=24.92-mu_ini=0.32-n0=20.00-gamma=0.0300-dt=0.00163\Quench\T=24.92-mu_final=0.08-n0=20.00-gamma=0.0300-dt=0.00163\metadata_final_state.json'


 70%|██████▉   | 17422/25000 [05:24<02:21, 53.71it/s]


KeyboardInterrupt: 